In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import time
import json
import random
from tqdm.auto import tqdm


In [ ]:
# Доли пропущенных признаков для экспериментов
MISSING_RATES = [0.0, 0.2, 0.5, 0.9]

# Настройка промпта для Heart
prompt_config = {
        "task": "Predict whether a patient has heart disease",
        "pos_label": "1",
        "neg_label": "0",
        "entity": "Patient",
        "question": "Does this patient have heart disease based on clinical features?"
}

base_output_dir = "/content/drive/MyDrive/finetuned_qwen_missing_Heart"

# 1. Загрузка данных

In [ ]:
import kagglehub
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def load_dataset():
    path = kagglehub.dataset_download("fedesoriano/heart-failure-prediction") + "/heart.csv"
    df = pd.read_csv(path)
    target_name = 'HeartDisease'
    X = df.drop(columns=[target_name])
    y = df[target_name]

    feature_names = X.columns.to_list()

    le = LabelEncoder()
    df[target_name] = le.fit_transform(df[target_name])

    return df, feature_names, target_name


def split_dataset(df, target_name, test_size=0.2, val_size=0.25, seed=42):
    # Разделение на train/val/test (60/20/20)
    train_val_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
        stratify=df[target_name]
    )

    train_df, val_df = train_test_split(
        train_val_df,
        test_size=val_size,
        random_state=seed,
        stratify=train_val_df[target_name]
    )

    return train_df, val_df, test_df

df, feature_names, target_name = load_dataset()
train_df, val_df, test_df = split_dataset(df, target_name)

Using Colab cache for faster access to the 'heart-failure-prediction' dataset.


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    object 
 2   ChestPainType   918 non-null    object 
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    object 
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    object 
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    object 
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 86.2+ KB


In [ ]:
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    counts = df[target_name].value_counts()
    pcts   = df[target_name].value_counts(normalize=True) * 100
    print(f"\n{name} (всего: {len(df)}):")
    print(f"  {prompt_config['neg_label']}: {counts[0]:5d} — {pcts[0]:.1f}%")
    print(f"  {prompt_config['pos_label']}: {counts[1]:5d} — {pcts[1]:.1f}%")


train (всего: 550):
  0:   246 — 44.7%
  1:   304 — 55.3%

val (всего: 184):
  0:    82 — 44.6%
  1:   102 — 55.4%

test (всего: 184):
  0:    82 — 44.6%
  1:   102 — 55.4%


# 2. Вспомогательные функции

In [ ]:
def row_to_text_template_with_missing(row, feature_names, target_name, missing_rate=0.0, seed=None):

    rng = np.random.default_rng(seed)

    # Определяем, какие признаки пропустить
    n_drop = int(len(feature_names) * missing_rate)
    dropped = set(rng.choice(feature_names, size=n_drop, replace=False)) if n_drop > 0 else set()

    template_parts = []

    for feature in feature_names:
        if feature in dropped:
            continue  # пропускаем признак

        value = row[feature]

        if isinstance(value, (int, np.integer)):
            phrase = f"The value of {feature} is {value}."
        elif isinstance(value, (float, np.floating)):
            phrase = f"The value of {feature} is {value:.2f}."
        else:
            phrase = f"The category of {feature} is {value}."

        template_parts.append(phrase)

    text = " ".join(template_parts)

    return text

# Тест
test_row = train_df.iloc[0]
display(train_df.head(1))
for rate in MISSING_RATES:
    text_output = row_to_text_template_with_missing(
        row=test_row,
        feature_names=feature_names,
        target_name=target_name,
        missing_rate=rate
    )

    print(f"Missing Rate: {rate * 100:.0f}%")
    print(text_output)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
839,35,F,ASY,138,183,0,Normal,182,N,1.4,Up,0


Missing Rate: 0%
The value of Age is 35. The category of Sex is F. The category of ChestPainType is ASY. The value of RestingBP is 138. The value of Cholesterol is 183. The value of FastingBS is 0. The category of RestingECG is Normal. The value of MaxHR is 182. The category of ExerciseAngina is N. The value of Oldpeak is 1.40. The category of ST_Slope is Up.
Missing Rate: 20%
The value of Age is 35. The category of Sex is F. The category of ChestPainType is ASY. The value of RestingBP is 138. The value of FastingBS is 0. The category of RestingECG is Normal. The value of MaxHR is 182. The value of Oldpeak is 1.40. The category of ST_Slope is Up.
Missing Rate: 50%
The category of Sex is F. The category of ChestPainType is ASY. The value of Cholesterol is 183. The value of FastingBS is 0. The value of MaxHR is 182. The category of ST_Slope is Up.
Missing Rate: 90%
The value of MaxHR is 182. The category of ST_Slope is Up.


In [ ]:
def parse_prediction(response, prompt_config):
    response = response.lower().strip().rstrip('.,!? ')
    pos, neg = prompt_config['pos_label'].lower(), prompt_config['neg_label'].lower()

    # Точное совпадение
    if response == pos: return 1
    if response == neg: return 0

    # Начинается с метки
    if response.startswith(pos): return 1
    if response.startswith(neg): return 0

    # Содержит метку как отдельное слово
    words = response.split()
    if pos in words: return 1
    if neg in words: return 0

    # Не удалось распознать
    print(f"Warning: Could not parse '{response}' (expected '{prompt_config['pos_label']}' or '{prompt_config['neg_label']}')")
    return 0



response = prompt_config['pos_label']
pred = parse_prediction(response, prompt_config)
print(f"Response: '{response}'\nPrediction: {pred}\n")

response = "hi"
pred = parse_prediction(response, prompt_config)
print(f"Response: '{response}'\nPrediction: {pred}")

Response: '1'
Prediction: 1

Response: 'hi'
Prediction: 0


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
# Вычисление метрик качества
def compute_metrics(y_true, y_pred, y_prob=None):
    roc = roc_auc_score(y_true, y_prob if y_prob is not None else y_pred)
    return {
        "ROC-AUC": roc_auc_score(y_true, y_prob if y_prob is not None else y_pred),
        "F1":       f1_score(y_true, y_pred, zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall":   recall_score(y_true, y_pred, zero_division=0),
    }

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils import resample

def bootstrap_metrics(y_true, y_pred, y_prob=None, n_iter=1000):
    scores = []

    for i in range(n_iter):
        # Bootstrap выборка
        if y_prob is not None:
            y_true_boot, y_pred_boot, y_prob_boot = resample(
                y_true, y_pred, y_prob, random_state=i+1
            )
        else:
            y_true_boot, y_pred_boot = resample(
                y_true, y_pred, random_state=i+1
            )
            y_prob_boot = None

        try:
            # Вычисление метрик
            if y_prob_boot is not None:
                auc = roc_auc_score(y_true_boot, y_prob_boot)
            else:
                auc = 0.0

            f1 = f1_score(y_true_boot, y_pred_boot, zero_division=0)
            pr = precision_score(y_true_boot, y_pred_boot, zero_division=0)
            rc = recall_score(y_true_boot, y_pred_boot, zero_division=0)

            acc = accuracy_score(y_true_boot, y_pred_boot)
            scores.append((auc, f1, acc, pr, rc))

        except ValueError:
            # Пропускание сэмплов, где не представлены все классы
            continue

    scores = np.asarray(scores)
    means, stds = scores.mean(0), scores.std(0, ddof=1)
    names = ["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"]

    return {n: f"{m:.4f}±{s:.4f}" for n, m, s in zip(names, means, stds)}

# Тестирование функции
sample_row = test_df.iloc[0]
sample_text = row_to_text_template_with_missing(sample_row, feature_names, target_name)
print(f"\nПример текста:\n{sample_text[:300]}")


Пример текста:
The value of Age is 46. The category of Sex is M. The category of ChestPainType is ASY. The value of RestingBP is 115. The value of Cholesterol is 0. The value of FastingBS is 0. The category of RestingECG is Normal. The value of MaxHR is 113. The category of ExerciseAngina is Y. The value of Oldpea


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

#Загрузка модели Qwen 3.0 4B Instruct
model_name = "Qwen/Qwen3-4B-Instruct-2507"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Загрузка токенизаторов
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Загрузка модель с 16-битной квантизацией и автоматическим распределением по устройствам
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map=device, # Explicitly force all model layers to the detected device (GPU if available)
    attn_implementation="sdpa"
)

if torch.cuda.is_available():
    print(f"Использовано памяти: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Использовано памяти: 8.04 GB


In [ ]:
def create_prompt_missing(row, feature_names, target_name, prompt_config, tokenizer, missing_rate=0.0, seed=None):

    system_prompt = (
        f"You are a classifier. {prompt_config['task']}: "
        f"'{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'. "
        f"Answer with only one word: '{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'."
    )

    input_text = row_to_text_template_with_missing(row, feature_names, target_name, missing_rate, seed)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",
         "content": f"{prompt_config['entity']} information: {input_text}\n{prompt_config['question']}"},
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True # добавление role assistant
    )

In [ ]:
import torch.nn.functional as F

def predict_single_with_prob(prompt, prompt_config, model, tokenizer, device, max_new_tokens=5):

    model_inputs = tokenizer([prompt], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            model_inputs.input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            output_scores=True,
            return_dict_in_generate=True
        )

    # Декодирование текста
    generated_ids = outputs.sequences[0][model_inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()

    # Извлечение вероятностей для меток из prompt_config
    first_token_logits = outputs.scores[0][0]

    pos_id = tokenizer.encode(prompt_config['pos_label'], add_special_tokens=False)[0]
    neg_id = tokenizer.encode(prompt_config['neg_label'], add_special_tokens=False)[0]

    pos_logit = first_token_logits[pos_id]
    neg_logit = first_token_logits[neg_id]

    probs = F.softmax(torch.stack([pos_logit, neg_logit]), dim=0)
    prob_pos = probs[0].item()

    return response, prob_pos

# Тест
test_prompt = create_prompt_missing(test_df.iloc[0], feature_names, target_name, prompt_config, tokenizer)
response, prob = predict_single_with_prob(test_prompt, prompt_config, base_model, tokenizer, device)
print(f"Response: '{response}', Probability: {prob:.4f}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Response: '1', Probability: 1.0000


# 3. Fine-tuning с LoRA

## Подготовка данных для Fine-tuning

In [ ]:
def balance_dataset(train_df, target_name, prompt_config, seed=42):
    df_0 = train_df[train_df[target_name] == 0]
    df_1 = train_df[train_df[target_name] == 1]

    df_majority = df_0 if len(df_0) > len(df_1) else df_1
    df_minority = df_1 if len(df_0) > len(df_1) else df_0

    print(f"До балансировки:")
    print(f"{prompt_config['neg_label']}:  {len(df_majority)}")
    print(f"{prompt_config['pos_label']}: {len(df_minority)}")

    df_minority_up = resample(df_minority, replace=True,
                              n_samples=len(df_majority), random_state=seed)

    train_df_balanced = pd.concat([df_majority, df_minority_up])
    train_df_balanced = train_df_balanced.sample(frac=1, random_state=seed).reset_index(drop=True)

    print(f"\nПосле балансировки:")
    print(train_df_balanced[target_name].value_counts())

    return train_df_balanced

train_df_balanced = balance_dataset(train_df, target_name, prompt_config)

До балансировки:
0:  304
1: 246

После балансировки:
HeartDisease
1    304
0    304
Name: count, dtype: int64


In [ ]:
# Создание текстов для обучения

def create_training_dataset(df, feature_names, target_name, prompt_config, tokenizer, missing_rate=0.0):
    system_prompt = (
        f"You are a classifier. {prompt_config['task']}: "
        f"'{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'. "
        f"Answer with only one word: '{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'."
    )
    texts = []

    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=f"Dataset (missing={missing_rate:.0%})")):
        input_text = row_to_text_template_with_missing(row, feature_names, target_name, missing_rate, idx)
        target = prompt_config['pos_label'] if row[target_name] == 1 else prompt_config['neg_label']

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"{prompt_config['entity']} information: {input_text}\n{prompt_config['question']}"},
            {"role": "assistant", "content": target}
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)

    return texts


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

def finetune_model(missing_rate, train_df_balanced, feature_names, target_name,
                   prompt_config, tokenizer, base_model, device,
                   num_epochs=3, batch_size=16):
    """
    Полный цикл fine-tuning для заданного missing_rate.
    Возвращает путь к лучшему checkpoint.
    """
    output_dir = f"{base_output_dir}_missing{int(missing_rate * 100):03d}"
    print(f"  Fine-tuning: missing_rate = {missing_rate:.0%}")
    print(f"  Output dir : {output_dir}")

    # Датасет
    train_texts = create_training_dataset(
        train_df_balanced, feature_names, target_name,
        prompt_config, tokenizer, missing_rate=missing_rate)
    train_dataset = Dataset.from_dict({"text": train_texts})

    def tokenize_fn(examples):
        return tokenizer(examples["text"], truncation=True, max_length=512, padding=False)

    tokenized_dataset = train_dataset.map(tokenize_fn, batched=True,
                                          remove_columns=train_dataset.column_names)

    # LoRA
    lora_config = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
    )
    model_lora = get_peft_model(base_model, lora_config)
    model_lora.print_trainable_parameters()

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        bf16=True,
        logging_steps=10,
        save_strategy="epoch",
        optim="adamw_torch_fused",
        warmup_steps=50,
        max_grad_norm=1.0,
        weight_decay=0.01,
        report_to="none",
        dataloader_num_workers=4,
        group_by_length=True,
    )

    trainer = Trainer(
        model=model_lora,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    )

    t0 = time.time()
    trainer.train()
    print(f"Обучение завершено за {(time.time()-t0)/60:.1f} мин")

    trainer.save_model(output_dir)

    # Возвращаем базовую модель без LoRA весов (важно для следующего эксперимента)
    model_lora = model_lora.unload()
    torch.cuda.empty_cache()

    return output_dir

Mounted at /content/drive


In [ ]:
import glob
import re
from peft import PeftModel

def evaluate_checkpoints_on_val(output_dir, val_df, feature_names, target_name,
                                 prompt_config, tokenizer, base_model, device,
                                 eval_missing_rate=0.0):

    checkpoints = sorted(
        glob.glob(f"{output_dir}/checkpoint-*"),
        key=lambda x: int(re.search(r'checkpoint-(\d+)', x).group(1))
    )
    print(f"Найдено {len(checkpoints)} checkpoints в {output_dir}")

    best_auc, best_checkpoint = 0.0, None
    for cp in checkpoints:
        model_eval = PeftModel.from_pretrained(base_model, cp).eval()

        y_true, y_prob = [], []
        for _, row in tqdm(val_df.iterrows(), total=len(val_df),
                           desc=f"Val eval {cp.split('/')[-1]}"):
            prompt = create_prompt_missing(
                row, feature_names, target_name, prompt_config, tokenizer,
                missing_rate=eval_missing_rate)
            _, prob = predict_single_with_prob(prompt, prompt_config, model_eval, tokenizer, device)
            y_true.append(row[target_name])
            y_prob.append(prob)

        y_pred = [1 if p > 0.5 else 0 for p in y_prob]
        auc = roc_auc_score(y_true, y_prob)
        print(f"  {cp.split('/')[-1]}  ROC-AUC={auc:.4f}")

        if auc > best_auc:
            best_auc = auc
            best_checkpoint = cp

        del model_eval
        torch.cuda.empty_cache()

    print(f"Лучший checkpoint: {best_checkpoint}  (ROC-AUC={best_auc:.4f})")
    return best_checkpoint

In [ ]:
def evaluate_on_test(best_checkpoint, test_df, feature_names, target_name,
                     prompt_config, tokenizer, base_model, device,
                     eval_missing_rate=0.0):
    model_finetuned = PeftModel.from_pretrained(base_model, best_checkpoint).eval()

    y_true, y_pred, y_prob = [], [], []
    t0 = time.time()

    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test eval"):
        prompt = create_prompt_missing(
            row, feature_names, target_name, prompt_config, tokenizer,
            missing_rate=eval_missing_rate)
        response, prob = predict_single_with_prob(
            prompt, prompt_config, model_finetuned, tokenizer, device)
        y_true.append(row[target_name])
        y_pred.append(parse_prediction(response, prompt_config))
        y_prob.append(prob)

    elapsed = time.time() - t0
    metrics = compute_metrics(np.array(y_true), np.array(y_pred), np.array(y_prob))
    boot = bootstrap_metrics(np.array(y_true), np.array(y_pred), np.array(y_prob), n_iter=1000)

    print("Метрики:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
    print("Bootstrap (mean±std):")
    for k, v in boot.items():
        print(f"  {k}: {v}")

    del model_finetuned
    torch.cuda.empty_cache()

    return {
        "metrics": metrics,
        "bootstrap": boot,
        "time_total": elapsed,
        "time_per_sample": elapsed / len(y_true),
        "best_checkpoint": best_checkpoint,
        "eval_missing_rate": eval_missing_rate,
    }

In [ ]:
all_results = {}

start_time = time.time()

for missing_rate in MISSING_RATES:
    tag = f"missing_{int(missing_rate * 100):03d}pct"
    print(f"Эксперимент: missing_rate = {missing_rate:.0%}")

    # 7.1 Fine-tuning
    output_dir = finetune_model(
        missing_rate=missing_rate,
        train_df_balanced=train_df_balanced,
        feature_names=feature_names,
        target_name=target_name,
        prompt_config=prompt_config,
        tokenizer=tokenizer,
        base_model=base_model,
        device=device,
        num_epochs=20,
        batch_size=16,
    )

    # 7.2 Выбор лучшего checkpoint по val
    best_cp = evaluate_checkpoints_on_val(
        output_dir=output_dir,
        val_df=val_df,
        feature_names=feature_names,
        target_name=target_name,
        prompt_config=prompt_config,
        tokenizer=tokenizer,
        base_model=base_model,
        device=device,
        eval_missing_rate=missing_rate,
    )

    # 7.3 Оценка на test
    result = evaluate_on_test(
        best_checkpoint=best_cp,
        test_df=test_df,
        feature_names=feature_names,
        target_name=target_name,
        prompt_config=prompt_config,
        tokenizer=tokenizer,
        base_model=base_model,
        device=device,
        eval_missing_rate=missing_rate,
    )

    all_results[tag] = result

print(f"Эксперимент завершен за {(time.time()-start_time)/60:.1f} мин")

Эксперимент: missing_rate = 0%
  Fine-tuning: missing_rate = 0%
  Output dir : /content/drive/MyDrive/finetuned_qwen_missing_Heart_missing000


Dataset (missing=0%):   0%|          | 0/608 [00:00<?, ?it/s]

Map:   0%|          | 0/608 [00:00<?, ? examples/s]

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


Step,Training Loss
10,1.954306
20,1.010748
30,0.269984
40,0.144365
50,0.140453
60,0.138765
70,0.136414
80,0.133298
90,0.134886
100,0.133373


...

Val eval checkpoint-380:   0%|          | 0/184 [00:00<?, ?it/s]

  checkpoint-380  ROC-AUC=0.7020
Лучший checkpoint: /content/drive/MyDrive/finetuned_qwen_missing_Heart_missing090/checkpoint-266  (ROC-AUC=0.7859)


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Test eval:   0%|          | 0/184 [00:00<?, ?it/s]

Метрики:
  ROC-AUC: 0.7707
  F1: 0.7500
  Accuracy: 0.7283
  Precision: 0.7653
  Recall: 0.7353
Bootstrap (mean±std):
  ROC-AUC: 0.7703±0.0362
  F1: 0.7482±0.0344
  Accuracy: 0.7274±0.0336
  Precision: 0.7644±0.0428
  Recall: 0.7343±0.0431
Эксперимент завершен за 86.0 мин


In [ ]:
all_results

{'missing_000pct': {'metrics': {'ROC-AUC': np.float64(0.9409971305595409),
   'F1': 0.892018779342723,
   'Accuracy': 0.875,
   'Precision': 0.8558558558558559,
   'Recall': 0.9313725490196079},
  'bootstrap': {'ROC-AUC': '0.9400±0.0172',
   'F1': '0.8904±0.0225',
   'Accuracy': '0.8737±0.0246',
   'Precision': '0.8548±0.0335',
   'Recall': '0.9301±0.0246'},
  'time_total': 40.70475125312805,
  'time_per_sample': 0.2212214742017829,
  'best_checkpoint': '/content/drive/MyDrive/finetuned_qwen_missing_Heart_missing000/checkpoint-152',
  'eval_missing_rate': 0.0},
 'missing_020pct': {'metrics': {'ROC-AUC': np.float64(0.9071018651362984),
   'F1': 0.837696335078534,
   'Accuracy': 0.8315217391304348,
   'Precision': 0.898876404494382,
   'Recall': 0.7843137254901961},
  'bootstrap': {'ROC-AUC': '0.9061±0.0233',
   'F1': '0.8366±0.0290',
   'Accuracy': '0.8312±0.0279',
   'Precision': '0.8983±0.0326',
   'Recall': '0.7841±0.0402'},
  'time_total': 40.643665075302124,
  'time_per_sample': 0.

In [ ]:
print("\nBootstrap (mean±std):")
for tag, res in all_results.items():
    rate = tag.replace("missing_", "").replace("pct", "%").lstrip("0") or "0%"
    print(f"\n  missing={rate}")
    for k, v in res["bootstrap"].items():
        print(f"    {k}: {v}")


Bootstrap (mean±std):

  missing=%
    ROC-AUC: 0.9400±0.0172
    F1: 0.8904±0.0225
    Accuracy: 0.8737±0.0246
    Precision: 0.8548±0.0335
    Recall: 0.9301±0.0246

  missing=20%
    ROC-AUC: 0.9061±0.0233
    F1: 0.8366±0.0290
    Accuracy: 0.8312±0.0279
    Precision: 0.8983±0.0326
    Recall: 0.7841±0.0402

  missing=50%
    ROC-AUC: 0.8608±0.0283
    F1: 0.7917±0.0320
    Accuracy: 0.7928±0.0297
    Precision: 0.8904±0.0343
    Recall: 0.7141±0.0437

  missing=90%
    ROC-AUC: 0.7703±0.0362
    F1: 0.7482±0.0344
    Accuracy: 0.7274±0.0336
    Precision: 0.7644±0.0428
    Recall: 0.7343±0.0431


ROC-AUC (0.94): Исключительный уровень производительности. Модель демонстрирует высокую точность в диагностике сердечных заболеваний. Даже при 50% пропусков показатель удерживается на уровне 0.86, что подтверждает способность LLM находить альтернативные паттерны в биометрических данных при дефиците информации.

Recall (0.93): Высочайший показатель полноты при полных данных. Модель успешно идентифицирует 93% пациентов с патологиями, что жизненно важно для медицинского скрининга. Примечательно, что при 90% пропусков полнота сохраняется на уровне 0.73, демонстрируя поразительную устойчивость к потере клинических признаков.

Precision (0.85): Высокая точность классификации. Модель минимизирует количество ложноположительных диагнозов. При 20% и 50% пропусков наблюдается рост точности до 0.89-0.90, что свидетельствует о том, что при частичном отсутствии данных модель становится более «строгой» и уверенной в своих положительных прогнозах.

Accuracy (0.87): Стабильная общая точность. Модель эффективно справляется с выявлением сердечно-сосудистых рисков. Снижение точности до 0.73 при 90% пропусков является закономерным, однако этот результат всё еще значительно превышает порог случайного классификатора.

Время эксперимента: ~ 1 ч 26 мин.

Использовано оперативная памяти графического процесса: 35/80GB.

Графический процессор A100 80GB.